# Module 12: Capstone, ship and demo

**Project:** turn your prototype into something you can **put in front of strangers**: harden it, work out whether the business adds up, write the technical report, and **pitch it in five minutes with a live demo**.

**Done when:**
- your prototype is **hardened** (bad input cannot crash it, errors do not leak, requests are logged safely) and **deployed at a public URL** that works,
- you have a **cost model** (cost per request, price, margin, break-even),
- you wrote a **short technical report** with data, model choice, evaluation and **honest limitations**,
- you have a **5-minute pitch** that follows the standard order, with a live demo,
- you completed an **honest self-assessment** against the rubric.

**How to use this notebook**
1. This is a **workbook** for your project, like Module 11. Replace every `YOUR CODE HERE`. The **Self-check** cells tell you if you got it right.
2. **Use the project you built in Module 11.** If you are practising, set `USE_DEMO = True` for the ready-made ticket-urgency example.
3. Stuck? Use the hint ladder: the hints below each step, then the course forum. Only after your own attempt, open the solution page.

**The mindset of this module:** in Module 11 you asked *does it work?* Now you ask *what happens when it does not?*, *does it pay?*, and *can I make a stranger believe it in five minutes?*

## Step 0: Setup (given)

In [ ]:
import json, time, math, re, hashlib
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import requests

USE_DEMO = False      # True: practise with the ready-made demo project
print("Ready.")

## Step 1: Harden it

A prototype works when you feed it good input. A product must also survive **bad input**, **crashes** and **attackers**. Two functions give you most of the protection.

**`validate_input(raw, max_chars=2000)`** returns `(ok, reason)`:
- `(False, reason)` if `raw` is **not a string**, is **empty or only spaces**, or is **longer than `max_chars`** after `strip()`,
- otherwise `(True, "")`.
(If your project takes images or other input, adapt the checks to your input: file type, size, dimensions.)

**`safe_predict(predict_fn, raw, log_fn=None, max_chars=2000)`** is the **safety wrapper** around your model. It must **never raise an exception**. It returns a dictionary:
- invalid input: `{"ok": False, "error": <the reason>}`. **Do not call the model.**
- success: `{"ok": True, "result": <what predict_fn returned>, "latency_ms": <milliseconds spent in predict_fn>}`.
- if `predict_fn` **raises an exception**: return `{"ok": False, "error": "the model could not process this input"}`. **Do not put the exception text in the error**: it can reveal internal details (file paths, library names, secrets). Instead, if `log_fn` is given, call `log_fn(f"{type(e).__name__}: {e}")` so that *you* can find the cause.

In [ ]:
def validate_input(raw, max_chars=2000):
    """Return (ok, reason)."""
    # YOUR CODE HERE
    return True, ""

def safe_predict(predict_fn, raw, log_fn=None, max_chars=2000):
    """Run the model safely. Never raises."""
    # YOUR CODE HERE
    return {"ok": True, "result": predict_fn(raw), "latency_ms": 0.0}

### Self-check: validation and the safety wrapper

In [ ]:
for _bad, _why in ((None, "None"), (12345, "a number"), (["a list"], "a list"), ("", "an empty string"), ("   \n\t ", "only whitespace"), ("x" * 2001, "2001 characters")):
    _ok, _reason = validate_input(_bad)
    assert _ok is False and isinstance(_reason, str) and _reason, f"validate_input should reject {_why} with a reason, but returned {(_ok, _reason)}."
assert validate_input("hello there") == (True, ""), "A normal text is valid and the reason is an empty string."
assert validate_input("x" * 2000) == (True, "") and validate_input("  hi  ", max_chars=2) == (True, ""), "The limit applies to the text after strip(), and exactly max_chars is allowed."

_seen, _logs = [], []
def _good(x): _seen.append(x); return "result:" + x
def _broken(x): raise KeyError("secret_api_key_123")
_r = safe_predict(_good, "  hello  ")
assert _r["ok"] is True and _r["result"] == "result:hello" and _r["latency_ms"] >= 0, f"A valid input should give ok=True, the result of predict_fn (called with the stripped text), and a latency. Got {_r}."
_n = len(_seen)
_r = safe_predict(_good, "")
assert _r["ok"] is False and "error" in _r and len(_seen) == _n, "Invalid input must give ok=False and must NOT call the model."
try:
    _r = safe_predict(_broken, "valid text", log_fn=_logs.append)
except Exception as e:
    raise AssertionError(f"safe_predict() must never raise, but it raised {type(e).__name__}: {e}")
assert _r["ok"] is False and "secret_api_key_123" not in json.dumps(_r), "The error returned to the user must not contain the exception text: it can reveal internal details."
assert len(_logs) == 1 and "KeyError" in _logs[0] and "secret_api_key_123" in _logs[0], "The details must go to log_fn instead, as 'ExceptionType: message', so that you can debug it."
_slow = safe_predict(lambda x: (time.sleep(0.05), "ok")[1], "text")
assert 45 <= _slow["latency_ms"] < 1000, f"latency_ms must be the milliseconds spent in the model (about 50 here) but is {_slow['latency_ms']}."
print("Hardening check passed: bad input is rejected, crashes are contained, and details stay in the log.")

Now **attack your own project**. This cell sends a set of nasty inputs through your safety wrapper. Add **your own** nasty inputs for your project (a huge input, the wrong language, an input that is valid but weird) and look at what happens. This cell is given.

In [ ]:
import sys, types
def build_demo_serve():
    # a stand-in model for practice, used when USE_DEMO is True
    def demo_predict(text):
        if "boom" in text:
            raise RuntimeError("secret internal detail: /srv/model/weights.bin missing")
        return {"output": "urgent" if "down" in text.lower() else "normal"}
    return demo_predict

predict_fn = build_demo_serve() if USE_DEMO else None      # YOUR CODE HERE (if not demo): the function that runs your model on one input
errors_log = []
nasty = [None, "", "   ", "a" * 5000, 12345, ["a", "b"], "\x00\x00 weird bytes", "SELECT * FROM users; --", "emoji only 😀😀😀", "boom", "our site is down"]
for x in nasty:
    out = safe_predict(predict_fn, x, log_fn=errors_log.append)
    shown = repr(x)[:30]
    print(f"{shown:34s} -> ok={out['ok']}  {out.get('error', str(out.get('result'))[:40])}")
print(f"\n{len(errors_log)} internal error(s) went to the log:", errors_log)

## Step 2: Does the business add up?

A product that costs more to run than it earns is a hobby. Write two small functions.

**`unit_economics(price_per_request, cost_per_request, fixed_monthly, requests_per_month)`** returns a dictionary with:
- `revenue`: price x requests,
- `variable_cost`: cost per request x requests,
- `gross_profit`: revenue minus variable cost,
- `gross_margin`: gross profit / revenue (0.0 if revenue is 0),
- `net_profit`: gross profit minus the fixed monthly costs,
- `break_even_requests`: the requests per month at which net profit is zero: `fixed_monthly / (price - cost)`. If `price <= cost`, return `float("inf")` (you lose money on every request).

**`price_for_margin(cost_per_request, target_gross_margin)`**: the price that gives a target gross margin. Since margin = (price - cost) / price, the price is `cost / (1 - margin)`.

In [ ]:
def unit_economics(price_per_request, cost_per_request, fixed_monthly, requests_per_month):
    # YOUR CODE HERE
    return {}

def price_for_margin(cost_per_request, target_gross_margin):
    # YOUR CODE HERE
    return 0.0

### Self-check: unit economics

In [ ]:
_u = unit_economics(0.01, 0.002, 50, 100_000)
assert isinstance(_u, dict) and "revenue" in _u, "unit_economics() must return a dictionary with revenue, variable_cost, gross_profit, gross_margin, net_profit and break_even_requests."
assert abs(_u["revenue"] - 1000) < 1e-6 and abs(_u["variable_cost"] - 200) < 1e-6 and abs(_u["gross_profit"] - 800) < 1e-6, f"At 0.01 per request and 0.002 cost over 100,000 requests: revenue 1000, variable cost 200, gross profit 800. You got {_u}."
assert abs(_u["gross_margin"] - 0.8) < 1e-9 and abs(_u["net_profit"] - 750) < 1e-6, f"gross margin should be 0.8 and net profit 750 (800 minus 50 fixed). You got {_u['gross_margin']} and {_u['net_profit']}."
assert abs(_u["break_even_requests"] - 6250) < 1e-6, f"break-even = fixed / (price - cost) = 50 / 0.008 = 6250 requests, but you got {_u['break_even_requests']}."
_loss = unit_economics(0.001, 0.002, 50, 1000)
assert _loss["break_even_requests"] == float("inf") and _loss["gross_profit"] < 0, "If the price is below the cost, you lose money on every request: break_even_requests must be infinity."
_zero = unit_economics(0.01, 0.002, 50, 0)
assert _zero["gross_margin"] == 0.0 and _zero["net_profit"] == -50, "With no requests: gross margin 0.0 (do not divide by zero) and net profit equal to minus the fixed costs."
assert abs(price_for_margin(0.002, 0.8) - 0.01) < 1e-12 and abs(price_for_margin(0.01, 0.5) - 0.02) < 1e-12, "price = cost / (1 - margin): a cost of 0.002 and a margin of 0.8 gives a price of 0.01."
print("unit_economics() check passed.")

Now build **three scenarios** for your product: **pessimistic**, **base** and **optimistic**. Use your **measured** cost per request from Module 10 (not a guess), a price you could justify to a customer (what do they pay today for the problem you solve?), your real fixed monthly costs, and a number of requests that you can defend. Then explain your price in `PRICING_RATIONALE`.

In [ ]:
ASSUMPTIONS = {
    "pessimistic": {"price_per_request": 0.0, "cost_per_request": 0.0, "fixed_monthly": 0.0, "requests_per_month": 0},
    "base":        {"price_per_request": 0.0, "cost_per_request": 0.0, "fixed_monthly": 0.0, "requests_per_month": 0},
    "optimistic":  {"price_per_request": 0.0, "cost_per_request": 0.0, "fixed_monthly": 0.0, "requests_per_month": 0},
}
PRICING_RATIONALE = """
YOUR CODE HERE: why this price? What does the customer pay today for this problem, and what margin do you need?
"""

### Self-check: scenarios

In [ ]:
assert set(ASSUMPTIONS) >= {"pessimistic", "base", "optimistic"}, "Define the three scenarios: pessimistic, base and optimistic."
for _name, _a in ASSUMPTIONS.items():
    assert set(_a) >= {"price_per_request", "cost_per_request", "fixed_monthly", "requests_per_month"}, f"Scenario {_name!r} needs price_per_request, cost_per_request, fixed_monthly and requests_per_month."
    assert _a["price_per_request"] > 0 and _a["cost_per_request"] > 0 and _a["requests_per_month"] > 0 and _a["fixed_monthly"] >= 0, f"In scenario {_name!r} price, cost and requests must be positive numbers (you cannot serve for free)."
_r = [ASSUMPTIONS[k]["requests_per_month"] for k in ("pessimistic", "base", "optimistic")]
assert _r[0] < _r[1] < _r[2], "The number of requests must grow from the pessimistic to the base to the optimistic scenario."
assert ASSUMPTIONS["pessimistic"]["price_per_request"] <= ASSUMPTIONS["base"]["price_per_request"] <= ASSUMPTIONS["optimistic"]["price_per_request"], "The price should not fall from the pessimistic to the optimistic scenario."
_p = PRICING_RATIONALE.lower()
assert "your code here" not in _p and len(PRICING_RATIONALE.strip()) >= 150, "Explain your price in at least two or three sentences (150 characters or more)."
assert any(w in _p for w in ("customer", "pays", "willing", "today", "alternative", "competitor")), "Anchor the price in what the customer pays or would pay today, not only in your costs."
assert "margin" in _p or "cost" in _p, "Mention your margin or your costs."
print("Scenarios check passed.")

In [ ]:
print(f"{'scenario':12s} {'revenue':>10s} {'gross margin':>13s} {'net profit':>11s} {'break-even requests':>20s}")
scenario_results = {}
for name, a in ASSUMPTIONS.items():
    u = unit_economics(a["price_per_request"], a["cost_per_request"], a["fixed_monthly"], a["requests_per_month"])
    scenario_results[name] = u
    be = "never" if u["break_even_requests"] == float("inf") else f"{u['break_even_requests']:,.0f}"
    print(f"{name:12s} {u['revenue']:10,.0f} {u['gross_margin']:13.1%} {u['net_profit']:11,.0f} {be:>20s}")
print("\nCost-plus price for a 80% gross margin on the base scenario:", f"{price_for_margin(ASSUMPTIONS['base']['cost_per_request'], 0.8):.5f}", "per request")

**Question:** which scenario loses money, and why? What would you have to believe for the product to be a business? What is the **one number** in your table that an investor would question first?

*Your answer:*

## Step 3: The technical report

A short **technical report** lets someone else understand, trust and continue your work. It is **not** a sales document: it is honest. Fill in the six sections in the dictionary `REPORT`. Each section needs **at least 150 characters**.

- `summary`: what you built, for whom, and the headline result.
- `data`: where it came from, how many examples, how labeled, how split, and how the test set is kept fixed.
- `model_choice`: what you used (pretrained, fine-tuned, API), **why**, and which **baseline or alternative** you compared it with.
- `evaluation`: the **numbers**: your metric on the fixed set, the baseline's, the target, plus latency and cost.
- `limitations`: write **at least three points**, each starting with `- ` or `1.`. Cover **failure cases** (where it goes wrong, with examples), **data gaps** (what your data does not cover), and **risks** (bias, privacy, misuse, cost). This is the section that earns trust.
- `next_steps`: what you would do next, in order.

In [ ]:
REPORT = {"summary": "", "data": "", "model_choice": "", "evaluation": "", "limitations": "", "next_steps": ""}

def render_report(report, title, path="REPORT.md"):
    """Write the report as a Markdown file."""
    parts = [f"# {title}: technical report\n"]
    for key in ("summary", "data", "model_choice", "evaluation", "limitations", "next_steps"):
        parts.append(f"## {key.replace('_', ' ').capitalize()}\n\n{report[key].strip()}\n")
    Path(path).write_text("\n".join(parts), encoding="utf-8")
    return path

### Self-check: the report

In [ ]:
_keys = ["summary", "data", "model_choice", "evaluation", "limitations", "next_steps"]
assert set(REPORT) >= set(_keys), f"REPORT needs the six sections {_keys}."
for _k in _keys:
    assert isinstance(REPORT[_k], str) and len(REPORT[_k].strip()) >= 150, f"Write at least 150 characters for the section {_k!r}."
assert any(ch.isdigit() for ch in REPORT["evaluation"]) and len(re.findall(r"\d", REPORT["evaluation"])) >= 4, "The evaluation section must contain numbers (your metric, the baseline's, the target, latency or cost)."
_m = REPORT["model_choice"].lower()
assert any(w in _m for w in ("baseline", "alternative", "compared", "instead of", "rather than")), "In model_choice, say which baseline or alternative you compared with, and why you chose what you chose."
_lim = REPORT["limitations"]
_items = re.findall(r"(?m)^\s*(?:[-*]|\d+[.)])\s+\S", _lim)
assert len(_items) >= 3, f"List at least three limitations, each on its own line starting with '- ' or '1.' (found {len(_items)})."
_l = _lim.lower()
assert "fail" in _l or "wrong" in _l or "error" in _l or "miss" in _l, "Describe your failure cases (where it goes wrong, with examples)."
assert "data" in _l, "Describe the gaps in your data (what it does not cover)."
assert any(w in _l for w in ("risk", "bias", "privacy", "fair", "misuse", "harm")), "Name at least one risk (bias, privacy, misuse, cost of a mistake)."
_path = render_report(REPORT, PROJECT_NAME if "PROJECT_NAME" in globals() else "My project")
assert Path(_path).exists() and Path(_path).read_text().count("## ") == 6, "REPORT.md should have six sections."
print("Report check passed. Saved REPORT.md.")

## Step 4: The five-minute pitch

A good pitch has an **order**, and the order is not "start with the model". Use this one:

1. `problem`: who hurts, and how much?
2. `demo`: show it **working, live**, before you explain it. Show, then tell.
3. `how_it_works`: the simple version, in one picture.
4. `results`: your numbers, against the baseline, honestly (including a limitation).
5. `cost_and_market`: what it costs, what you charge, who pays.
6. `next_steps`: what you need and what you will do next.

Write `check_pitch_timing(slides, limit_seconds=300)`. `slides` is a list of dictionaries, each with a `"section"` and `"seconds"`. Return a dictionary with:
- `total_seconds`: the sum of all the slides' seconds,
- `within_limit`: `True` if the total is at most `limit_seconds`,
- `order_ok`: `True` if the list of section names **equals** `REQUIRED_ORDER` exactly (same sections, same order),
- `missing`: the list of sections from `REQUIRED_ORDER` that are not in `slides`,
- `longest`: the name of the section with the most seconds.

In [ ]:
REQUIRED_ORDER = ["problem", "demo", "how_it_works", "results", "cost_and_market", "next_steps"]

def check_pitch_timing(slides, limit_seconds=300):
    """Check the pitch against the time limit and the standard order."""
    # YOUR CODE HERE
    return {}

### Self-check: check_pitch_timing

In [ ]:
_good = [{"section": n, "seconds": s} for n, s in zip(REQUIRED_ORDER, (40, 90, 50, 50, 40, 30))]
_r = check_pitch_timing(_good)
assert isinstance(_r, dict) and "total_seconds" in _r, "check_pitch_timing() must return a dictionary with total_seconds, within_limit, order_ok, missing and longest."
assert _r["total_seconds"] == 300 and _r["within_limit"] is True and _r["order_ok"] is True and _r["missing"] == [] and _r["longest"] == "demo", f"For a 300-second pitch in the right order: total 300, within limit, order ok, nothing missing, longest 'demo'. You got {_r}."
_long = _good + [{"section": "extra", "seconds": 10}]
_r2 = check_pitch_timing(_long)
assert _r2["total_seconds"] == 310 and _r2["within_limit"] is False and _r2["order_ok"] is False, "310 seconds is over the 300-second limit, and an extra section means the order is not the required one."
_swap = [_good[1], _good[0]] + _good[2:]
assert check_pitch_timing(_swap)["order_ok"] is False, "The same sections in a different order must not count as the right order."
_miss = [s for s in _good if s["section"] != "results"]
assert check_pitch_timing(_miss)["missing"] == ["results"], "missing must list the required sections that are absent."
print("check_pitch_timing() check passed.")

Now write **your pitch**: one entry per section, in the required order, with a `title`, 2 or more `points` (the things you will say, one short line each), and the `seconds` you will spend. The **demo gets the most time**. Rehearse it out loud with a timer: the seconds you write must be the seconds you really take.

In [ ]:
PITCH = []      # one dict per section: {"section": "problem", "title": "...", "points": ["...", "..."], "seconds": 40}

### Self-check: your pitch

In [ ]:
assert len(PITCH) >= 1 and all(isinstance(s, dict) and {"section", "title", "points", "seconds"} <= set(s) for s in PITCH), "Each pitch entry needs section, title, points and seconds."
_t = check_pitch_timing(PITCH)
assert _t["order_ok"], f"Your sections must be exactly {REQUIRED_ORDER}, in that order. Missing: {_t['missing']}."
assert _t["within_limit"], f"Your pitch is {_t['total_seconds']} seconds, over the 300-second limit. Cut something."
assert _t["total_seconds"] >= 240, f"Your pitch is only {_t['total_seconds']} seconds. A five-minute slot should be used (at least 240 seconds): you probably have too little in it."
assert all(isinstance(s["points"], list) and len(s["points"]) >= 2 and all(len(p.strip()) >= 10 for p in s["points"]) for s in PITCH), "Each section needs at least two points of a full sentence each (10 characters or more)."
_demo = [s for s in PITCH if s["section"] == "demo"][0]
assert _demo["seconds"] >= 60, "The live demo is the heart of the pitch: give it at least 60 seconds."
assert _t["longest"] == "demo", f"The demo should be the longest part of your pitch, but {_t['longest']!r} is longer. Show more, explain less."
print(f"Pitch check passed: {_t['total_seconds']} seconds, in the right order, demo longest.")

## Step 5: The README

The README is the front door of your repository. Someone who finds it should understand **what it is**, **try it in a few minutes**, and **see what it can and cannot do**. Write `README_MD`, using Markdown headings (`## ...`). It must contain these sections:

- **What it does** (two sentences) and a **link to the live demo**,
- **How to run** (numbered steps, with the installation command),
- **Results** (your headline numbers, including the baseline),
- **Limitations** (honest, shortened from your report).

In [ ]:
README_MD = """
YOUR CODE HERE: the README of your project, in Markdown.
"""

### Self-check: the README

In [ ]:
_r = README_MD
_l = _r.lower()
assert "your code here" not in _l and len(_r.strip()) >= 400, "Write a real README (at least about 400 characters)."
assert re.search(r"(?m)^#\s+\S", _r), "Start with a title: a line beginning with '# '."
assert len(re.findall(r"(?m)^##\s+\S", _r)) >= 3, "Use at least three '## ' section headings (How to run, Results, Limitations)."
for _section in ("how to run", "results", "limitations"):
    assert re.search(rf"(?m)^##\s+.*{_section}", _l), f"Add a section whose heading contains {_section!r}."
assert re.search(r"(?m)^\s*1[.)]\s+\S", _r) and re.search(r"(?m)^\s*3[.)]\s+\S", _r), "How to run needs numbered steps (1. 2. 3.)."
assert "pip install" in _l or "requirements" in _l, "Show the installation command (for example pip install -r requirements.txt)."
assert "http" in _l, "Include a link to your live demo."
assert re.search(r"\d\.\d+|\d+%", _r), "The results section needs numbers."
Path("README.md").write_text(README_MD, encoding="utf-8")
print("README check passed. Saved README.md.")

## Step 6: Is it really online?

"It works on my machine" does not count. Put your demo at a **public URL** (a Hugging Face Space, as in Module 10, or any public address), then **verify it**. Set `PUBLIC_URL`, then run the given verification, which opens the address 20 times and reports the status codes and the time taken. It also saves your **final record**.

In [ ]:
PUBLIC_URL = ""      # YOUR CODE HERE: the https:// address of your deployed demo

In [ ]:
def verify_deployment(url, n=20, timeout=30):
    """Open the URL n times (after one warm-up request) and report status codes and timing."""
    try:
        requests.get(url, timeout=timeout)                        # warm-up: a sleeping Space needs a moment to wake
    except requests.RequestException:
        pass
    codes, ms = [], []
    for _ in range(n):
        t0 = time.perf_counter()
        try:
            codes.append(requests.get(url, timeout=timeout).status_code)
        except requests.RequestException:
            codes.append(0)
        ms.append((time.perf_counter() - t0) * 1000)
    return {"url": url, "n": n, "ok": sum(c == 200 for c in codes), "codes": sorted(set(codes)),
            "avg_ms": float(np.mean(ms)), "p95_ms": float(np.percentile(ms, 95))}

assert PUBLIC_URL.startswith("https://"), "PUBLIC_URL must be the full https:// address of your deployed demo."
deployment = verify_deployment(PUBLIC_URL)
print(f"{deployment['ok']} of {deployment['n']} requests answered 200   (average {deployment['avg_ms']:.0f} ms, p95 {deployment['p95_ms']:.0f} ms, codes {deployment['codes']})")
final_record = {"public_url": PUBLIC_URL, "deployment_check": deployment, "unit_economics": scenario_results,
                "pitch_seconds": check_pitch_timing(PITCH)["total_seconds"], "report_file": "REPORT.md", "readme_file": "README.md"}
Path("final_record.json").write_text(json.dumps(final_record, indent=2, default=str))
print("Saved final_record.json")

### Self-check: it is online

In [ ]:
assert deployment["ok"] >= 0.95 * deployment["n"], (f"Only {deployment['ok']} of {deployment['n']} requests succeeded (status codes seen: {deployment['codes']}). "
    "Open the address in a private browser window. Is the Space public, running (not sleeping or still building), and the address spelled exactly right?")
print("Deployment check passed: your demo answers from the public internet.")

**Do the stranger test:** send the address to someone who has never seen your project and watch them use it without helping. Write down the first thing that confused them:

*Your notes:*

## Step 7: An honest self-assessment

Before others judge your work, judge it yourself, **honestly**. Score each criterion from **1 to 5** and write the **evidence** for the score in at least a full sentence. An honest assessment names weaknesses: a project with no weaknesses has not been looked at.

| criterion | what a 5 looks like |
|---|---|
| `problem` | a specific user with a painful problem, backed by evidence |
| `working_demo` | works live on real inputs, handles bad input, public URL |
| `evaluation` | fixed test set, baseline, honest numbers against a target |
| `limitations` | specific failure cases, data gaps and risks, not generic disclaimers |
| `economics` | measured cost, defended price, scenarios, break-even |
| `pitch` | right order, demo first, within time, rehearsed |

In [ ]:
CRITERIA = ["problem", "working_demo", "evaluation", "limitations", "economics", "pitch"]
SELF_ASSESSMENT = {c: {"score": 0, "evidence": ""} for c in CRITERIA}     # YOUR CODE HERE: score 1 to 5, and the evidence

### Self-check: the self-assessment

In [ ]:
assert set(SELF_ASSESSMENT) == set(CRITERIA), f"Assess exactly these criteria: {CRITERIA}."
for _c, _a in SELF_ASSESSMENT.items():
    assert isinstance(_a["score"], int) and 1 <= _a["score"] <= 5, f"The score for {_c!r} must be a whole number from 1 to 5."
    assert isinstance(_a["evidence"], str) and len(_a["evidence"].strip()) >= 60, f"Write the evidence for {_c!r} in at least a full sentence (60 characters or more): what did you do, and what is still weak?"
assert min(a["score"] for a in SELF_ASSESSMENT.values()) <= 4, "Name at least one real weakness: if every criterion is a perfect 5, you have not looked critically."
_total = sum(a["score"] for a in SELF_ASSESSMENT.values())
print(f"Self-assessment check passed. Total {_total} out of {5 * len(CRITERIA)}. Your weakest criterion: {min(SELF_ASSESSMENT, key=lambda c: SELF_ASSESSMENT[c]['score'])}.")

## Demo day checklist

- [ ] **Rehearsed out loud, with a timer**, at least three times. Under 5 minutes.
- [ ] The **demo URL is open in a browser tab** before you start, and the Space is **awake** (open it a minute before).
- [ ] You know **exactly which input** you will paste in the demo, and you tested it **today**.
- [ ] A **backup plan** for when the network fails: a short screen recording or screenshots of the demo, ready to play.
- [ ] You can answer: *what is your baseline? What does it get wrong? What does it cost? Who pays?*
- [ ] You say **one honest limitation** yourself, before anyone asks. It builds trust.
- [ ] Your **README, report and repository link** are ready to share.

## Stretch challenge: open-source it properly

Publish your repository so that a stranger can use it: a **clear README** (Step 5), a **license** (for example MIT), a `requirements.txt` with pinned versions, a short `CONTRIBUTING` note, and **no secrets or private data** in the history (check carefully: a leaked key in an old commit is still leaked). Ask one person to clone it fresh and run it using only your README. Where did they get stuck?

*Your notes:*

## Reflection and submission

Write three lines in your portfolio repo's `module-12/README.md`:
1. Your project, its public URL, and the one-sentence pitch
2. Your headline result against the baseline, and your biggest limitation
3. What you would do in your first month if this were a real company

Then download this notebook (*File > Download > .ipynb*), push it to `module-12` together with `REPORT.md`, `README.md` and `final_record.json` (do **not** push private data or keys), and submit the link in Moodle. Present on demo day, and give your peers **kind, specific, useful** feedback.